# For 250506 Denmark Demonstration 

In [ ]:
import rby1_sdk as rby
from dynamixel_sdk import PortHandler, PacketHandler
import time
import numpy as np
import functions as fn

np.set_printoptions(suppress=True, precision=5, linewidth=300)  

SIM_OR_REAL = 2         # 0: Simulation / 1: Real Robot(Wifi mode) / 2: Real Robot (UPC mode)

In [ ]:
if (SIM_OR_REAL == 0):
    ROBOT_ADDRESS = "localhost:50051"           # For Docker Simulation
elif (SIM_OR_REAL == 1 or SIM_OR_REAL == 2):
    ROBOT_ADDRESS = "192.168.30.1:50051"        # For Real Robot
else:
    ROBOT_ADDRESS = "localhost:50051"           # For Safety, Default=Docker Simulation

robot = rby.create_robot_a(ROBOT_ADDRESS)       # Robot Model is "A"
model = robot.model()
if not robot.connect():
    print("Could not connect to robot")
    exit(1)
else:
    print("SUCCESS: Connect to the robot.")
    if (robot.power_on(".*") == True):             # Turn on all the power (5v, 12v, 24v, 48v)
        time.sleep(1)
        if (robot.servo_on(".*") == True):         # Enable servo motors on all actuators
            time.sleep(1)
            robot.reset_fault_control_manager()
            robot.enable_control_manager(unlimited_mode_enabled=True)
            print("SUCCESS: Power ON the Robot.")
            
            # Grippers 12V Power ON 
            time.sleep(1)
            robot.set_tool_flange_output_voltage("right", 12)
            robot.set_tool_flange_output_voltage("left", 12)
            print("SUCCESS: Power ON the Gripper.")
        else:
            print("Failed to power ON the Servo")

SUCCESS: Connect to the robot.
SUCCESS: Power ON the Robot.
SUCCESS: Power ON the Gripper.


In [4]:
if (SIM_OR_REAL == 2):
    # --- Open Gripper ports ---
    port_hdlr = PortHandler(fn.DEVICE_NAME_GRIPPER)
    packet_hdlr = PacketHandler(fn.PROTOCOL_VERSION)

    if port_hdlr.openPort():
        print(f"SUCCESS: open the port: {fn.DEVICE_NAME_GRIPPER}")
    else:
        raise RuntimeError(f"Failed to open the port: {fn.DEVICE_NAME_GRIPPER}")

    if port_hdlr.setBaudRate(fn.BAUDRATE):
        print(f"SUCCESS: change the baudrate {fn.BAUDRATE}")
    else:
        raise RuntimeError(f"Failed to change the baudrate")
    
    # --- Ping Test ---
    dxl_activated = []

    for dxl_id in [fn.R_DXL_ID, fn.L_DXL_ID]:
        dxl_model_num, dxl_comm_res, dxl_err = packet_hdlr.ping(port_hdlr, dxl_id)
        
        if dxl_comm_res != fn.COMM_SUCCESS:
            print("[DXL ID:%d] %s" % (dxl_id, packet_hdlr.getTxRxResult(dxl_comm_res)))
        elif dxl_err != 0:
            print("[DXL ID:%d] %s" % (dxl_id, packet_hdlr.getRxPacketError(dxl_err)))
        else:
            dxl_activated.append(dxl_id)
            print("[DXL ID:%d] ping Succeeded. Dynamixel model number : %d" % (dxl_id, dxl_model_num))
    
    # --- Enable Current Control ---
    for dxl_id in dxl_activated:
        fn.DXL_write_torque_mode(dxl_id, fn.TORQUE_DISABLE, port_hdlr, packet_hdlr)             # Operation mode 변경 전 torque OFF
        fn.DXL_write_current_limit(dxl_id, fn.CURRENT_LIMIT_MAX_A, port_hdlr, packet_hdlr)      # Current limit setting before torque ON
        fn.DXL_write_operation_mode(dxl_id, fn.CURRENT_CONTROL_MODE, port_hdlr, packet_hdlr)
        fn.DXL_write_torque_mode(dxl_id, fn.TORQUE_ENABLE, port_hdlr, packet_hdlr)
        


SerialException: [Errno 2] could not open port /dev/rby1_gripper: [Errno 2] No such file or directory: '/dev/rby1_gripper'

# Gripper Test

In [ ]:
fn.DXL_write_current_ampere_safety(fn.R_DXL_ID, 0.02, port_hdlr, packet_hdlr)
fn.DXL_write_current_ampere_safety(fn.L_DXL_ID, 0.02, port_hdlr, packet_hdlr)
time.sleep(5)
fn.DXL_write_current_ampere_safety(fn.R_DXL_ID, -0.02, port_hdlr, packet_hdlr)
fn.DXL_write_current_ampere_safety(fn.L_DXL_ID, -0.02, port_hdlr, packet_hdlr)

# Power OFF

In [33]:
robot.set_tool_flange_output_voltage("right", 0)
robot.set_tool_flange_output_voltage("left", 0)
time.sleep(1)
robot.disable_control_manager()
robot.power_off(".*")

True

# Ready Posture

In [ ]:
Desired_Pos = fn.READY_DEMO_POS      # ZERO_POS, READY_POS, PACK_POS, READY_DEMO_POS
fn.move_position(robot, Desired_Pos, minimum_time=5)
time.sleep(1)
fn.move_head(robot, fn.HEAD_POS_DEMO, minimum_time=2.0)
time.sleep(1)
fn.DXL_write_current_ampere_safety(fn.L_DXL_ID, fn.DEMO_ZEROING_CURRENT_L, port_hdlr, packet_hdlr)

[DXL ID:1] SUCCESS: Set Current [0.02 A]


# Go Forward to 1st Table

In [4]:
fn.move_forward_distance_smooth(robot, 0.45)         # Forward 30cm
time.sleep(1)

moved=0.000, speed=0.000
moved=0.000, speed=0.002
moved=0.000, speed=0.004
moved=0.000, speed=0.006
moved=0.000, speed=0.008
moved=0.000, speed=0.010
moved=0.000, speed=0.012
moved=0.001, speed=0.014
moved=0.001, speed=0.016
moved=0.001, speed=0.018
moved=0.002, speed=0.020
moved=0.002, speed=0.022
moved=0.003, speed=0.024
moved=0.003, speed=0.026
moved=0.004, speed=0.028
moved=0.005, speed=0.030
moved=0.005, speed=0.032
moved=0.006, speed=0.034
moved=0.007, speed=0.036
moved=0.008, speed=0.038
moved=0.009, speed=0.040
moved=0.010, speed=0.042
moved=0.011, speed=0.044
moved=0.012, speed=0.046
moved=0.014, speed=0.048
moved=0.015, speed=0.050
moved=0.016, speed=0.052
moved=0.017, speed=0.054
moved=0.019, speed=0.056
moved=0.020, speed=0.058
moved=0.022, speed=0.060
moved=0.024, speed=0.062
moved=0.025, speed=0.064
moved=0.027, speed=0.066
moved=0.028, speed=0.068
moved=0.030, speed=0.070
moved=0.032, speed=0.072
moved=0.034, speed=0.074
moved=0.036, speed=0.076
moved=0.038, speed=0.078


# Check Vision

In [8]:
obj_pos = fn.get_block_position_5sec()
T_left_ee_from_torso5 = fn.Get_T_left_ee_from_torso5(robot)

x_vision = obj_pos[0]
y_vision = obj_pos[1]
z_vision = obj_pos[2]

x_left_ee_from_torso5_now = T_left_ee_from_torso5[0,3]
y_left_ee_from_torso5_now = T_left_ee_from_torso5[1,3]

x_delta_robot = z_vision * np.cos(fn.HEAD_POS_DEMO[1])
x_delta_robot = x_delta_robot - x_left_ee_from_torso5_now - fn.EE_OFFSET_X_DEMO

y_delta_robot = -x_vision - y_left_ee_from_torso5_now
pos_delta_from_torso_L = [x_delta_robot, y_delta_robot, 0.0]

print(f"pos_delta_from_torso_L: {pos_delta_from_torso_L}")

X=-0.029, Y=0.122, Z=0.778
X=-0.031, Y=0.129, Z=0.818
X=-0.029, Y=0.122, Z=0.778
X=-0.030, Y=0.126, Z=0.801
X=-0.030, Y=0.124, Z=0.788
X=-0.029, Y=0.119, Z=0.754

===== Average =====
X=-0.030, Y=0.125, Z=0.792
pos_delta_from_torso_L: [np.float64(0.21539954055229743), np.float64(-0.20559175203433622), 0.0]


# Move Arms to blocks

In [31]:
# pos_delta_from_torso_R = [0.175, 0.14, 0.0]
pos_delta_from_torso_R = [0.0, 0.0, 0.0]
# pos_delta_from_torso_L = [0.18, -0.1, 0.0]
fn.move_arm_globalpos_both(robot, pos_delta_from_torso_R, pos_delta_from_torso_L)

time.sleep(1)
# fn.DXL_current_control_smooth(fn.R_DXL_ID, port_hdlr, packet_hdlr, fn.DEMO_LAY_CURRENT_R, fn.DEMO_LAY_CURRENT_END)
fn.DXL_current_control_smooth(fn.L_DXL_ID, port_hdlr, packet_hdlr, fn.DEMO_LAY_CURRENT_L, fn.DEMO_LAY_CURRENT_END)

T_target_R
: [[-0.      -0.34202 -0.93969  0.36553]
 [ 0.9962  -0.0819   0.0298  -0.23554]
 [-0.08715 -0.93612  0.34072 -0.09757]
 [ 0.       0.       0.       1.     ]]
T_target_L
: [[ 0.       0.34201 -0.93969  0.577  ]
 [-0.9962  -0.08189 -0.02981  0.08588]
 [-0.08715  0.93612  0.34071 -0.09757]
 [ 0.       0.       0.       1.     ]]
[DXL ID:1] SUCCESS: Set Starting Current [-0.02 A]


In [32]:
z = -0.1
# pos_delta_from_torso_R = [0.0, 0.0, z]
pos_delta_from_torso_R = [0.0, 0.0, 0.0]
pos_delta_from_torso_L = [0.0, 0.0, z]
fn.move_arm_globalpos_both(robot, pos_delta_from_torso_R, pos_delta_from_torso_L)
time.sleep(1)

T_target_R
: [[ 0.00001 -0.34203 -0.93969  0.36553]
 [ 0.9962  -0.08189  0.02981 -0.23554]
 [-0.08715 -0.93611  0.34073 -0.09757]
 [ 0.       0.       0.       1.     ]]
T_target_L
: [[ 0.       0.34202 -0.93969  0.577  ]
 [-0.9962  -0.08189 -0.02981  0.08588]
 [-0.08715  0.93612  0.34072 -0.19757]
 [ 0.       0.       0.       1.     ]]


# Catch Blocks

In [13]:
# fn.DXL_write_current_ampere_safety(fn.R_DXL_ID, fn.DEMO_GRIP_CURRENT_R, port_hdlr, packet_hdlr)
fn.DXL_write_current_ampere_safety(fn.L_DXL_ID, fn.DEMO_GRIP_CURRENT_L, port_hdlr, packet_hdlr)

[DXL ID:1] SUCCESS: Set Current [0.6 A]


# Lift UP

In [14]:
z = 0.22
# pos_delta_from_torso_R = [0.0, 0.0, z]
pos_delta_from_torso_R = [0.0, 0.0, 0.04]
pos_delta_from_torso_L = [0.0, 0.0, z]
fn.move_arm_globalpos_both(robot, pos_delta_from_torso_R, pos_delta_from_torso_L)
time.sleep(1)

T_target_R
: [[-0.      -0.34203 -0.93969  0.36553]
 [ 0.9962  -0.08188  0.0298  -0.23554]
 [-0.08714 -0.93612  0.34073 -0.05757]
 [ 0.       0.       0.       1.     ]]
T_target_L
: [[-0.00002  0.34181 -0.93977  0.51812]
 [-0.99619 -0.08194 -0.02978  0.16454]
 [-0.08718  0.93619  0.34052  0.02251]
 [ 0.       0.       0.       1.     ]]


# Move Backward

In [15]:
fn.move_forward_distance_smooth(robot, -0.45)         # Backward 30cm
time.sleep(1)

moved=0.000, speed=0.000
moved=0.000, speed=0.002
moved=0.000, speed=0.004
moved=0.000, speed=0.006
moved=0.000, speed=0.008
moved=0.000, speed=0.010
moved=0.000, speed=0.012
moved=0.001, speed=0.014
moved=0.001, speed=0.016
moved=0.001, speed=0.018
moved=0.002, speed=0.020
moved=0.002, speed=0.022
moved=0.003, speed=0.024
moved=0.003, speed=0.026
moved=0.004, speed=0.028
moved=0.005, speed=0.030
moved=0.005, speed=0.032
moved=0.006, speed=0.034
moved=0.007, speed=0.036
moved=0.008, speed=0.038
moved=0.009, speed=0.040
moved=0.010, speed=0.042
moved=0.011, speed=0.044
moved=0.012, speed=0.046
moved=0.013, speed=0.048
moved=0.015, speed=0.050
moved=0.016, speed=0.052
moved=0.017, speed=0.054
moved=0.019, speed=0.056
moved=0.020, speed=0.058
moved=0.022, speed=0.060
moved=0.023, speed=0.062
moved=0.025, speed=0.064
moved=0.026, speed=0.066
moved=0.028, speed=0.068
moved=0.030, speed=0.070
moved=0.032, speed=0.072
moved=0.034, speed=0.074
moved=0.035, speed=0.076
moved=0.037, speed=0.078


# Turn Right 

In [16]:
fn.rotate_inplace_smooth(robot, fn.TURN_RIGHT_90)                # Right turn 90degree
time.sleep(1)

theta_deg=0.00, error_deg=-83.50, rate=0.000
theta_deg=0.00, error_deg=-83.50, rate=-0.003
theta_deg=0.00, error_deg=-83.50, rate=-0.006
theta_deg=0.00, error_deg=-83.50, rate=-0.009
theta_deg=0.00, error_deg=-83.50, rate=-0.012
theta_deg=-0.00, error_deg=-83.50, rate=-0.015
theta_deg=-0.02, error_deg=-83.48, rate=-0.018
theta_deg=-0.03, error_deg=-83.47, rate=-0.021
theta_deg=-0.03, error_deg=-83.47, rate=-0.024
theta_deg=-0.03, error_deg=-83.47, rate=-0.027
theta_deg=-0.05, error_deg=-83.45, rate=-0.030
theta_deg=-0.07, error_deg=-83.43, rate=-0.033
theta_deg=-0.08, error_deg=-83.42, rate=-0.036
theta_deg=-0.11, error_deg=-83.39, rate=-0.039
theta_deg=-0.14, error_deg=-83.36, rate=-0.042
theta_deg=-0.18, error_deg=-83.32, rate=-0.045
theta_deg=-0.23, error_deg=-83.27, rate=-0.048
theta_deg=-0.28, error_deg=-83.22, rate=-0.051
theta_deg=-0.35, error_deg=-83.15, rate=-0.054
theta_deg=-0.42, error_deg=-83.08, rate=-0.057
theta_deg=-0.49, error_deg=-83.01, rate=-0.060
theta_deg=-0.58, er

# Go Forward to 2nd Table

In [17]:
fn.move_forward_distance_smooth(robot, 2.5)         # Forward 250cm
time.sleep(1)

moved=0.000, speed=0.000
moved=0.000, speed=0.002
moved=0.000, speed=0.004
moved=0.000, speed=0.006
moved=0.000, speed=0.008
moved=0.000, speed=0.010
moved=0.000, speed=0.012
moved=0.001, speed=0.014
moved=0.001, speed=0.016
moved=0.001, speed=0.018
moved=0.002, speed=0.020
moved=0.002, speed=0.022
moved=0.003, speed=0.024
moved=0.003, speed=0.026
moved=0.004, speed=0.028
moved=0.005, speed=0.030
moved=0.005, speed=0.032
moved=0.006, speed=0.034
moved=0.007, speed=0.036
moved=0.008, speed=0.038
moved=0.009, speed=0.040
moved=0.010, speed=0.042
moved=0.011, speed=0.044
moved=0.012, speed=0.046
moved=0.013, speed=0.048
moved=0.015, speed=0.050
moved=0.016, speed=0.052
moved=0.017, speed=0.054
moved=0.019, speed=0.056
moved=0.020, speed=0.058
moved=0.022, speed=0.060
moved=0.023, speed=0.062
moved=0.025, speed=0.064
moved=0.027, speed=0.066
moved=0.028, speed=0.068
moved=0.030, speed=0.070
moved=0.032, speed=0.072
moved=0.034, speed=0.074
moved=0.036, speed=0.076
moved=0.038, speed=0.078


# Turn Left

In [18]:
fn.rotate_inplace_smooth(robot, fn.TURN_LEFT_90)                # Left turn 90degree
time.sleep(1)

theta_deg=0.00, error_deg=83.50, rate=0.000
theta_deg=0.00, error_deg=83.50, rate=0.003
theta_deg=0.00, error_deg=83.50, rate=0.006
theta_deg=0.00, error_deg=83.50, rate=0.009
theta_deg=0.00, error_deg=83.50, rate=0.012
theta_deg=0.00, error_deg=83.50, rate=0.015
theta_deg=0.01, error_deg=83.49, rate=0.018
theta_deg=0.02, error_deg=83.48, rate=0.021
theta_deg=0.04, error_deg=83.46, rate=0.024
theta_deg=0.05, error_deg=83.45, rate=0.027
theta_deg=0.06, error_deg=83.44, rate=0.030
theta_deg=0.08, error_deg=83.42, rate=0.033
theta_deg=0.12, error_deg=83.38, rate=0.036
theta_deg=0.15, error_deg=83.35, rate=0.039
theta_deg=0.20, error_deg=83.30, rate=0.042
theta_deg=0.25, error_deg=83.25, rate=0.045
theta_deg=0.31, error_deg=83.19, rate=0.048
theta_deg=0.37, error_deg=83.13, rate=0.051
theta_deg=0.44, error_deg=83.06, rate=0.054
theta_deg=0.51, error_deg=82.99, rate=0.057
theta_deg=0.58, error_deg=82.92, rate=0.060
theta_deg=0.65, error_deg=82.85, rate=0.063
theta_deg=0.73, error_deg=82.77,

# Move Forward

In [ ]:
fn.move_forward_distance_smooth(robot, 0.30)         # Forward 30cm
time.sleep(1)

moved=0.000, speed=0.000
moved=0.000, speed=0.002
moved=0.000, speed=0.004
moved=0.000, speed=0.006
moved=0.000, speed=0.008
moved=0.000, speed=0.010
moved=0.000, speed=0.012
moved=0.000, speed=0.014
moved=0.001, speed=0.016
moved=0.001, speed=0.018
moved=0.002, speed=0.020
moved=0.002, speed=0.022
moved=0.003, speed=0.024
moved=0.003, speed=0.026
moved=0.004, speed=0.028
moved=0.005, speed=0.030
moved=0.005, speed=0.032
moved=0.006, speed=0.034
moved=0.007, speed=0.036
moved=0.008, speed=0.038
moved=0.009, speed=0.040
moved=0.010, speed=0.042
moved=0.011, speed=0.044
moved=0.012, speed=0.046
moved=0.013, speed=0.048
moved=0.015, speed=0.050
moved=0.016, speed=0.052
moved=0.018, speed=0.054
moved=0.019, speed=0.056
moved=0.020, speed=0.058
moved=0.022, speed=0.060
moved=0.023, speed=0.062
moved=0.025, speed=0.064
moved=0.027, speed=0.066
moved=0.028, speed=0.068
moved=0.030, speed=0.070
moved=0.032, speed=0.072
moved=0.034, speed=0.074
moved=0.036, speed=0.076
moved=0.038, speed=0.078


# Move Arms to Collecting Boxes

In [20]:
# pos_delta_from_torso_R = [0.0, -0.17, 0.0]
pos_delta_from_torso_R = [0.0, 0.0, 0.0]
pos_delta_from_torso_L = [0.0, 0.13, 0.0]
fn.move_arm_globalpos_both(robot, pos_delta_from_torso_R, pos_delta_from_torso_L)
time.sleep(1)

T_target_R
: [[-0.      -0.34203 -0.93969  0.36553]
 [ 0.9962  -0.08189  0.0298  -0.23554]
 [-0.08714 -0.93611  0.34073 -0.05758]
 [ 0.       0.       0.       1.     ]]
T_target_L
: [[-0.00002  0.34182 -0.93977  0.51812]
 [-0.99619 -0.08191 -0.02978  0.29454]
 [-0.08716  0.93619  0.34052  0.02251]
 [ 0.       0.       0.       1.     ]]


In [ ]:
z = -0.06
# pos_delta_from_torso_R = [0.0, 0.0, z]
pos_delta_from_torso_R = [0.0, 0.0, 0.0]
pos_delta_from_torso_L = [0.0, 0.0, z]
fn.move_arm_globalpos_both(robot, pos_delta_from_torso_R, pos_delta_from_torso_L)

T_target_R
: [[-0.      -0.34203 -0.93969  0.36553]
 [ 0.9962  -0.08189  0.0298  -0.23554]
 [-0.08714 -0.93611  0.34073 -0.05758]
 [ 0.       0.       0.       1.     ]]
T_target_L
: [[-0.00002  0.34179 -0.93978  0.51812]
 [-0.99619 -0.08193 -0.02977  0.29454]
 [-0.08717  0.9362   0.34049 -0.07749]
 [ 0.       0.       0.       1.     ]]


True

# Lay blocks

In [22]:
# fn.DXL_current_control_smooth(fn.R_DXL_ID, port_hdlr, packet_hdlr, fn.DEMO_LAY_CURRENT_R, fn.DEMO_LAY_CURRENT_END)
fn.DXL_current_control_smooth(fn.L_DXL_ID, port_hdlr, packet_hdlr, fn.DEMO_LAY_CURRENT_L, fn.DEMO_LAY_CURRENT_END)

[DXL ID:1] SUCCESS: Set Starting Current [-0.02 A]


# Move Arms

In [23]:
z = 0.06
# pos_delta_from_torso_R = [0.0, 0.0, z]
pos_delta_from_torso_R = [0.0, 0.0, 0.0]
pos_delta_from_torso_L = [0.0, 0.0, z]
fn.move_arm_globalpos_both(robot, pos_delta_from_torso_R, pos_delta_from_torso_L)
time.sleep(1)

T_target_R
: [[-0.      -0.34203 -0.93969  0.36553]
 [ 0.9962  -0.08188  0.0298  -0.23554]
 [-0.08714 -0.93612  0.34072 -0.05758]
 [ 0.       0.       0.       1.     ]]
T_target_L
: [[-0.00003  0.34181 -0.93977  0.51812]
 [-0.99619 -0.08194 -0.02977  0.29454]
 [-0.08718  0.93619  0.34051 -0.01749]
 [ 0.       0.       0.       1.     ]]


# Move Backward

In [24]:
fn.move_forward_distance_smooth(robot, -0.30)         # Backward 30cm
time.sleep(1)

moved=0.000, speed=0.000
moved=0.000, speed=0.002
moved=0.000, speed=0.004
moved=0.000, speed=0.006
moved=0.000, speed=0.008
moved=0.000, speed=0.010
moved=0.000, speed=0.012
moved=0.001, speed=0.014
moved=0.001, speed=0.016
moved=0.001, speed=0.018
moved=0.002, speed=0.020
moved=0.002, speed=0.022
moved=0.003, speed=0.024
moved=0.003, speed=0.026
moved=0.004, speed=0.028
moved=0.005, speed=0.030
moved=0.005, speed=0.032
moved=0.006, speed=0.034
moved=0.007, speed=0.036
moved=0.008, speed=0.038
moved=0.009, speed=0.040
moved=0.010, speed=0.042
moved=0.011, speed=0.044
moved=0.012, speed=0.046
moved=0.013, speed=0.048
moved=0.015, speed=0.050
moved=0.016, speed=0.052
moved=0.017, speed=0.054
moved=0.018, speed=0.056
moved=0.020, speed=0.058
moved=0.021, speed=0.060
moved=0.023, speed=0.062
moved=0.024, speed=0.064
moved=0.026, speed=0.066
moved=0.028, speed=0.068
moved=0.029, speed=0.070
moved=0.031, speed=0.072
moved=0.033, speed=0.074
moved=0.035, speed=0.076
moved=0.037, speed=0.078


# Turn Left

In [25]:
fn.rotate_inplace_smooth(robot, fn.TURN_LEFT_90)                # Left turn 90degree
time.sleep(1)

theta_deg=0.00, error_deg=83.50, rate=0.000
theta_deg=0.00, error_deg=83.50, rate=0.003
theta_deg=0.00, error_deg=83.50, rate=0.006
theta_deg=0.00, error_deg=83.50, rate=0.009
theta_deg=0.00, error_deg=83.50, rate=0.012
theta_deg=0.00, error_deg=83.50, rate=0.015
theta_deg=0.01, error_deg=83.49, rate=0.018
theta_deg=0.03, error_deg=83.47, rate=0.021
theta_deg=0.04, error_deg=83.46, rate=0.024
theta_deg=0.05, error_deg=83.45, rate=0.027
theta_deg=0.06, error_deg=83.44, rate=0.030
theta_deg=0.07, error_deg=83.43, rate=0.033
theta_deg=0.09, error_deg=83.41, rate=0.036
theta_deg=0.12, error_deg=83.38, rate=0.039
theta_deg=0.15, error_deg=83.35, rate=0.042
theta_deg=0.20, error_deg=83.30, rate=0.045
theta_deg=0.25, error_deg=83.25, rate=0.048
theta_deg=0.31, error_deg=83.19, rate=0.051
theta_deg=0.37, error_deg=83.13, rate=0.054
theta_deg=0.42, error_deg=83.08, rate=0.057
theta_deg=0.47, error_deg=83.03, rate=0.060
theta_deg=0.53, error_deg=82.97, rate=0.063
theta_deg=0.60, error_deg=82.90,

# Move Forward to 1st Table

In [26]:
fn.move_forward_distance_smooth(robot, 2.5)         # Forward 250cm
time.sleep(1)

moved=0.000, speed=0.000
moved=0.000, speed=0.002
moved=0.000, speed=0.004
moved=0.000, speed=0.006
moved=0.000, speed=0.008
moved=0.000, speed=0.010
moved=0.000, speed=0.012
moved=0.000, speed=0.014
moved=0.001, speed=0.016
moved=0.001, speed=0.018
moved=0.002, speed=0.020
moved=0.002, speed=0.022
moved=0.003, speed=0.024
moved=0.003, speed=0.026
moved=0.004, speed=0.028
moved=0.005, speed=0.030
moved=0.006, speed=0.032
moved=0.006, speed=0.034
moved=0.007, speed=0.036
moved=0.008, speed=0.038
moved=0.009, speed=0.040
moved=0.010, speed=0.042
moved=0.011, speed=0.044
moved=0.012, speed=0.046
moved=0.013, speed=0.048
moved=0.015, speed=0.050
moved=0.016, speed=0.052
moved=0.017, speed=0.054
moved=0.019, speed=0.056
moved=0.020, speed=0.058
moved=0.022, speed=0.060
moved=0.023, speed=0.062
moved=0.025, speed=0.064
moved=0.027, speed=0.066
moved=0.028, speed=0.068
moved=0.030, speed=0.070
moved=0.032, speed=0.072
moved=0.034, speed=0.074
moved=0.036, speed=0.076
moved=0.038, speed=0.078


# Turn Right

In [27]:
fn.rotate_inplace_smooth(robot, fn.TURN_RIGHT_90)                # Right turn 90degree
time.sleep(1)

theta_deg=0.00, error_deg=-83.50, rate=0.000
theta_deg=0.00, error_deg=-83.50, rate=-0.003
theta_deg=0.00, error_deg=-83.50, rate=-0.006
theta_deg=0.00, error_deg=-83.50, rate=-0.009
theta_deg=0.00, error_deg=-83.50, rate=-0.012
theta_deg=0.00, error_deg=-83.50, rate=-0.015
theta_deg=-0.00, error_deg=-83.50, rate=-0.018
theta_deg=-0.00, error_deg=-83.50, rate=-0.021
theta_deg=-0.00, error_deg=-83.50, rate=-0.024
theta_deg=-0.02, error_deg=-83.48, rate=-0.027
theta_deg=-0.03, error_deg=-83.47, rate=-0.030
theta_deg=-0.06, error_deg=-83.44, rate=-0.033
theta_deg=-0.09, error_deg=-83.41, rate=-0.036
theta_deg=-0.12, error_deg=-83.38, rate=-0.039
theta_deg=-0.17, error_deg=-83.33, rate=-0.042
theta_deg=-0.23, error_deg=-83.27, rate=-0.045
theta_deg=-0.28, error_deg=-83.22, rate=-0.048
theta_deg=-0.34, error_deg=-83.16, rate=-0.051
theta_deg=-0.39, error_deg=-83.11, rate=-0.054
theta_deg=-0.45, error_deg=-83.05, rate=-0.057
theta_deg=-0.51, error_deg=-82.99, rate=-0.060
theta_deg=-0.58, err

In [ ]:
# right_deltas = [
#     [0, -0.1, 0],
#     [0, 0.1, 0],
#     [0, 0, 0.1],
#     [0, 0, -0.1]
# ]

# stream = robot.create_command_stream()
# for delta in right_deltas:
#     stream.send_command(fn.move_endeffector(robot, fn.RIGHT_END, delta))
#     time.sleep(2.0)
# stream.cancel()
# stream.wait()

In [ ]:
# pos_delta = [0, 0.0, 0.1]
# fn.move_arm_pos(robot, fn.LEFT_END, pos_delta)

In [ ]:
# pos_delta = [0.24, -0.17, 0.1]
# rpy_delta = [0.0, 0.0, np.deg2rad(90.0)]
# fn.move_arm_6D_relpos_relori(robot, fn.LEFT_END, pos_delta, rpy_delta)

In [ ]:
# pos_delta = [0.1, -0.1, 0.05]
# pos_delta = [0, 0, 0]
# rpy_from_torso = [0, 0, 0]
# fn.move_arm_6D_relpos_globalori(robot, fn.LEFT_END, pos_delta, rpy_from_torso)

In [ ]:
# pos_delta = [0.2, 0, 0]
# fn.move_arm_relpos(robot, fn.LEFT_END, pos_delta)

In [ ]:
# # 0.2 / -0.06 / 
# pos_delta_from_torso = [0.23, -0.06, 0.0]
# fn.move_arm_globalpos(robot, fn.LEFT_END, pos_delta_from_torso)

In [ ]:
# x = -0.03
# pos_delta_from_torso_R = [x, 0.0, 0.0]
# pos_delta_from_torso_L = [x, 0.0, 0.0]
# fn.move_arm_globalpos_both(robot, pos_delta_from_torso_R, pos_delta_from_torso_L)

T_target_R
: [[-0.00005 -0.34197 -0.93971  0.51552]
 [ 0.99622 -0.08166  0.02966 -0.09555]
 [-0.08688 -0.93616  0.34068 -0.09757]
 [ 0.       0.       0.       1.     ]]
T_target_L
: [[-0.00005  0.34186 -0.93975  0.51552]
 [-0.99623 -0.08155 -0.02961  0.09554]
 [-0.08676  0.9362   0.34058 -0.09755]
 [ 0.       0.       0.       1.     ]]


True

In [ ]:
# z = -0.12
# pos_delta_from_torso_R = [0.0, 0.0, z]
# pos_delta_from_torso_L = [0.0, 0.0, z]
# fn.move_arm_globalpos_both(robot, pos_delta_from_torso_R, pos_delta_from_torso_L)

T_target_R
: [[-0.00001 -0.34215 -0.93965  0.5155 ]
 [ 0.99621 -0.08174  0.02976 -0.09556]
 [-0.08699 -0.93608  0.34085 -0.19757]
 [ 0.       0.       0.       1.     ]]
T_target_L
: [[ 0.00002  0.34214 -0.93965  0.5155 ]
 [-0.99621 -0.08169 -0.02977  0.09556]
 [-0.08694  0.93609  0.34084 -0.19757]
 [ 0.       0.       0.       1.     ]]


True

In [ ]:
# z = 0.04
# pos_delta_from_torso_R = [0.0, 0.0, z]
# pos_delta_from_torso_L = [0.0, 0.0, z]
# fn.move_arm_globalpos_both(robot, pos_delta_from_torso_R, pos_delta_from_torso_L)

T_target_R
: [[ 0.01077 -0.36802 -0.92976  0.51047]
 [ 0.99709 -0.06627  0.03778 -0.10029]
 [-0.07552 -0.92745  0.36623 -0.17246]
 [ 0.       0.       0.       1.     ]]
T_target_L
: [[ 0.0026   0.35765 -0.93385  0.51132]
 [-0.99655 -0.07652 -0.03208  0.09754]
 [-0.08293  0.93072  0.35622 -0.16606]
 [ 0.       0.       0.       1.     ]]


True

In [ ]:
# pos_delta_from_torso_R = [0.0, 0.0, -0.13]
# pos_delta_from_torso_L = [0.0, 0.0, -0.13]
# fn.move_arm_globalpos_both(robot, pos_delta_from_torso_R, pos_delta_from_torso_L)

In [ ]:
# rpy_from_torso = [0, 0, 0]
# fn.move_arm_globalori(robot, fn.LEFT_END, rpy_from_torso)